In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque

# ==========================================
# 1. 定義 DQN 神經網路架構 (Policy Network)
# ==========================================
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        # 輸入層：[風險偏好, 市場趨勢, 價格變動百分比]
        # 輸出層：[Q_buy, Q_hold, Q_sell]
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 32)
        self.relu2 = nn.ReLU()
        self.out = nn.Linear(32, output_dim)

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        return self.out(x)

# ==========================================
# 2. 定義模擬環境 (Stock Trading Environment)
# ==========================================
class StockEnv:
    def __init__(self):
        self.state_dim = 3 # risk_level(1-10), market_trend(0,1,2), price_change_pct(-20~20)
        self.action_dim = 3 # 0: Buy, 1: Hold, 2: Sell
        self.reset()

    def reset(self):
        # 隨機初始化狀態
        self.risk_level = float(random.randint(1, 10))
        self.market_trend = float(random.choice([0, 1, 2])) # 0:Bear, 1:Neutral, 2:Bull
        self.price_change_pct = random.uniform(-20.0, 20.0)
        return np.array([self.risk_level, self.market_trend, self.price_change_pct], dtype=np.float32)

    def step(self, action):
        # 根據 UI 的定價公式，我們設定合理的獎勵函數讓 AI 學習
        # UI 買入閥值：下跌 (11 - risk_level) %
        # UI 賣出閥值：上漲 (5 + risk_level) %

        optimal_buy_drop = -(11 - self.risk_level)
        optimal_sell_rise = (5 + self.risk_level)

        reward = 0.0

        if action == 0: # Buy
            if self.price_change_pct <= optimal_buy_drop:
                reward = 10.0 + abs(self.price_change_pct - optimal_buy_drop) # 買在安全邊際內給予高獎勵
            else:
                reward = -5.0 # 太早買給予懲罰

        elif action == 2: # Sell
            if self.price_change_pct >= optimal_sell_rise:
                reward = 10.0 + (self.price_change_pct - optimal_sell_rise) # 風險溢酬耗盡時賣出
            else:
                reward = -5.0 # 太早賣給予懲罰

        elif action == 1: # Hold
            if optimal_buy_drop < self.price_change_pct < optimal_sell_rise:
                reward = 5.0 # 在觀望區間內 Hold 是好選擇
            else:
                reward = -2.0

        # 移動到下一個隨機狀態 (這裡簡化為獨立事件)
        next_state = self.reset()
        done = True # 簡化為一步 MDP 讓模型快速收斂到特定閥值

        return next_state, reward, done

# ==========================================
# 3. 經驗回放池 (Replay Buffer)
# ==========================================
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done

    def __len__(self):
        return len(self.buffer)

# ==========================================
# 4. 訓練流程 (Training Loop)
# ==========================================
def train_dqn():
    env = StockEnv()
    model = DQN(env.state_dim, env.action_dim)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    replay_buffer = ReplayBuffer(10000)

    batch_size = 64
    gamma = 0.99
    epsilon = 1.0
    epsilon_decay = 0.995
    epsilon_min = 0.01
    episodes = 5000

    print("開始訓練 DQN 模型...")

    for episode in range(episodes):
        state = env.reset()
        done = False
        total_reward = 0

        while not done:
            # Epsilon-greedy 策略
            if random.random() < epsilon:
                action = random.randint(0, 2)
            else:
                with torch.no_grad():
                    q_values = model(torch.tensor(state, dtype=torch.float32))
                    action = q_values.argmax().item()

            next_state, reward, done = env.step(action)
            replay_buffer.push(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward

            if len(replay_buffer) > batch_size:
                # 採樣並訓練
                b_state, b_action, b_reward, b_next_state, b_done = replay_buffer.sample(batch_size)

                b_state = torch.tensor(b_state, dtype=torch.float32)
                b_action = torch.tensor(b_action, dtype=torch.int64).unsqueeze(1)
                b_reward = torch.tensor(b_reward, dtype=torch.float32).unsqueeze(1)
                b_next_state = torch.tensor(b_next_state, dtype=torch.float32)
                b_done = torch.tensor(b_done, dtype=torch.float32).unsqueeze(1)

                # 計算當前 Q 值
                q_values = model(b_state).gather(1, b_action)

                # 計算目標 Q 值
                with torch.no_grad():
                    next_q_values = model(b_next_state).max(1)[0].unsqueeze(1)
                    target_q_values = b_reward + gamma * next_q_values * (1 - b_done)

                loss = loss_fn(q_values, target_q_values)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        if (episode + 1) % 1000 == 0:
            print(f"Episode {episode + 1}/{episodes} | Epsilon: {epsilon:.3f} | Loss: {loss.item():.4f}")

    print("✅ 模型訓練完成！")
    return model

# ==========================================
# 5. 執行訓練並匯出 ONNX 模型 (供前端使用)
# ==========================================
if __name__ == "__main__":
    trained_model = train_dqn()

    # 自動安裝 Colab 缺少的 ONNX 相關套件
    try:
        import onnxscript
    except ImportError:
        print("🔧 偵測到缺少 onnxscript，正在自動安裝必要套件 (這可能需要幾秒鐘)...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx", "onnxscript"])
        print("✅ 套件安裝完成！繼續匯出模型...")

    # 匯出 ONNX 模型
    export_path = "dqn_policy.onnx"
    dummy_input = torch.randn(1, 3, dtype=torch.float32) # [1, 3] 是輸入的 Tensor 維度

    print(f"正在匯出模型至 {export_path} ...")
    torch.onnx.export(
        trained_model,               # 訓練好的 PyTorch 模型
        dummy_input,                 # 模型的 Dummy input
        export_path,                 # 輸出的檔名
        export_params=True,          # 儲存訓練好的權重
        opset_version=11,            # ONNX 版本 (建議用 11 兼容性最好)
        do_constant_folding=True,    # 最佳化常量折疊
        input_names=['input_state'], # 輸入層名稱
        output_names=['q_values'],   # 輸出層名稱
        dynamic_axes={'input_state': {0: 'batch_size'}, 'q_values': {0: 'batch_size'}}
    )
    print("✅ ONNX 模型匯出成功！")

    # 在 Colab 中自動觸發下載
    try:
        from google.colab import files
        files.download(export_path)
        print("📥 瀏覽器已開始下載模型檔案...")
    except ImportError:
        print("提示：非 Colab 環境，請直接在資料夾中尋找 dqn_policy.onnx")

開始訓練 DQN 模型...
Episode 1000/5000 | Epsilon: 0.010 | Loss: 5.9982
Episode 2000/5000 | Epsilon: 0.010 | Loss: 3.3609
Episode 3000/5000 | Epsilon: 0.010 | Loss: 4.4437
Episode 4000/5000 | Epsilon: 0.010 | Loss: 2.4285
Episode 5000/5000 | Epsilon: 0.010 | Loss: 1.0600
✅ 模型訓練完成！
🔧 偵測到缺少 onnxscript，正在自動安裝必要套件 (這可能需要幾秒鐘)...
✅ 套件安裝完成！繼續匯出模型...
正在匯出模型至 dqn_policy.onnx ...


/tmp/ipykernel_7031/4088021684.py:185: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
/tmp/ipykernel_7031/4088021684.py:185: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0512 17:50:58.654000 7031 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_versi

[torch.onnx] Obtain model graph for `DQN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DQN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
✅ ONNX 模型匯出成功！


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 瀏覽器已開始下載模型檔案...
